# Agente de Investigación de Mercado con Bing Search

## Descripción

En este ejercicio, crearás un **agente de IA especializado en investigación de mercado** utilizando el SDK de Agentes de IA de Azure con integración de **Bing Search**. El agente buscará información actualizada en Internet sobre tendencias tecnológicas, análisis de competidores, noticias de la industria y datos de mercado en tiempo real.

### Casos de Uso
- **Investigación de tendencias**: Buscar las últimas tendencias en tecnología, IA, cloud computing, etc.
- **Análisis competitivo**: Obtener información actualizada sobre competidores y productos del mercado
- **Noticias de la industria**: Recopilar noticias recientes sobre sectores específicos
- **Investigación de productos**: Buscar información sobre productos, servicios o tecnologías específicas

### Herramientas Utilizadas
- **Bing Grounding Tool**: Permite al agente realizar búsquedas en Internet y obtener información actualizada
- **Azure AI Agents SDK**: Para crear y gestionar el agente

## Prerrequisitos

Asegúrate de que tu archivo `.env` en la raíz del repositorio contiene las siguientes variables:
- `PROJECT_ENDPOINT`: El endpoint de tu proyecto de Azure AI Foundry
- `MODEL_DEPLOYMENT_NAME`: El nombre del modelo desplegado
- `BING_CONNECTION_ID`: El ID de la conexión de Bing Search configurada en Azure AI Foundry

### Configurar la conexión de Bing Search en Azure AI Foundry

Antes de ejecutar este notebook, necesitas crear una conexión de Bing Search en Azure AI Foundry:

1. Ve a tu proyecto en Azure AI Foundry Portal
2. Navega a **Settings** > **Connections**
3. Crea una nueva conexión de tipo **Bing Search**
4. Proporciona tu API Key de Bing Search
5. Copia el **Connection ID** y añádelo a tu archivo `.env`

### Cargar librerías y variables de entorno

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

# Import required classes from Azure AI Agents SDK
from azure.identity import DefaultAzureCredential
from azure.ai.agents import AgentsClient
from azure.ai.agents.models import BingGroundingTool, ListSortOrder, MessageRole

# Load environment variables from .env file
load_dotenv(find_dotenv(usecwd=True))
project_endpoint = os.getenv("PROJECT_ENDPOINT")
model_deployment = os.getenv("MODEL_DEPLOYMENT_NAME")
bing_connection_id = os.getenv("BING_CONNECTION_ID")

# Verificar que las variables de entorno estén configuradas
if not all([project_endpoint, model_deployment, bing_connection_id]):
    raise ValueError(
        "Error: Asegúrate de que PROJECT_ENDPOINT, MODEL_DEPLOYMENT_NAME y BING_CONNECTION_ID "
        "estén configurados en tu archivo .env"
    )

print("Variables de entorno cargadas correctamente")
print(f"Endpoint: {project_endpoint}")
print(f"Modelo: {model_deployment}")
print(f"Bing Connection ID: {bing_connection_id}")

### Conectar al Cliente de Agentes (Agent Client)

Nos conectamos al proyecto de Azure AI Foundry usando las credenciales de Azure.

In [ ]:
print("Conectando al Azure AI Agents Client...")
agent_client = AgentsClient(
    endpoint=project_endpoint,
    credential=DefaultAzureCredential(
        exclude_environment_credential=True,
        exclude_managed_identity_credential=True
    )
)
print("Cliente de agentes conectado exitosamente")

### Crear la herramienta Bing Grounding Tool

Creamos una herramienta `BingGroundingTool` que permite al agente buscar información en Internet usando Bing Search.

**Parámetros de BingGroundingTool:**
- `connection_id`: ID de la conexión de Bing Search configurada en Azure AI Foundry
- `count`: Número máximo de resultados de búsqueda (por defecto: 5)
- `freshness`: Filtro de frescura de los resultados ('day', 'week', 'month', o vacío para todos)
- `market`: Código de mercado/región (ej: 'es-ES' para España, 'en-US' para Estados Unidos)
- `set_lang`: Idioma de la interfaz de usuario (ej: 'es', 'en')

In [ ]:
print("Creando Bing Grounding Tool...")

# Crear la herramienta de búsqueda de Bing con configuración en español
bing_tool = BingGroundingTool(
    connection_id=bing_connection_id,
    count=5,  # Número de resultados de búsqueda
    market="es-ES",  # Mercado español
    set_lang="es",  # Idioma español
    freshness="week"  # Resultados de la última semana
)

print("Bing Grounding Tool creada exitosamente")

### Definir un agente que usa Bing Search

Creamos un agente de IA especializado en investigación de mercado con instrucciones específicas sobre su rol y capacidades.

In [ ]:
print("Creando agente de investigación de mercado...")

agent = agent_client.create_agent(
    model=model_deployment,
    name="market-research-agent",
    instructions="""
Eres un agente de IA especializado en investigación de mercado y análisis de tendencias tecnológicas.

Tu rol es:
1. Buscar información actualizada en Internet usando Bing Search
2. Analizar tendencias tecnológicas y de mercado
3. Recopilar noticias relevantes de la industria
4. Proporcionar análisis competitivos y de productos
5. Presentar la información de forma estructurada y profesional

Cuando respondas:
- Usa búsquedas en Internet para obtener información actualizada
- Cita las fuentes de información cuando sea posible
- Proporciona análisis claros y concisos
- Organiza la información en secciones
- Destaca los puntos clave y tendencias importantes

Responde siempre en español de forma profesional y bien estructurada.
""",
    tools=bing_tool.definitions,
)

print(f"Agente creado: {agent.name} (id: {agent.id})")

### Crear un hilo (thread) para la conversación

Iniciamos un hilo donde se ejecutará la sesión de chat con el agente.

In [ ]:
print("Creando thread...")
thread = agent_client.threads.create()
print(f"Thread creado (id: {thread.id})")

### Definir la lógica para enviar prompts y procesar respuestas

Esta función encapsula la lógica para enviar un mensaje al agente, procesarlo y obtener la respuesta.

In [ ]:
def send_prompt_to_agent(prompt):
    print(f"\n{'='*80}")
    print(f"Enviando prompt: '{prompt}'")
    print(f"{'='*80}\n")
    
    message = agent_client.messages.create(
        thread_id=thread.id,
        role="user",
        content=prompt,
    )
    print(f"Mensaje de usuario creado (thread: {thread.id})")

    # Crear y procesar la ejecución (run)
    print("Procesando solicitud (esto puede tardar unos momentos mientras busca en Internet)...")
    run = agent_client.runs.create_and_process(thread_id=thread.id, agent_id=agent.id)
    print(f"Ejecución completada con estado: {run.status}")

    # Comprobar el estado de la ejecución
    if run.status == "failed":
        print(f"Error en la ejecución: {run.last_error}")
    else:
        print("\nObteniendo respuesta del agente...\n")
        # Mostrar la última respuesta del agente
        last_msg = agent_client.messages.get_last_message_text_by_role(
            thread_id=thread.id,
            role=MessageRole.AGENT,
        )
        if last_msg:
            print("RESPUESTA DEL AGENTE:")
            print("-" * 80)
            print(f"{last_msg.text.value}")
            print("-" * 80)

print("Función 'send_prompt_to_agent' definida exitosamente")

## Ejemplos de Uso del Agente de Investigación de Mercado

A continuación, veremos varios ejemplos de cómo usar el agente para diferentes casos de uso de investigación de mercado.

### Ejemplo 1: Investigar tendencias tecnológicas

In [ ]:
user_prompt = "¿Cuáles son las principales tendencias en inteligencia artificial y machine learning en 2025?"

send_prompt_to_agent(user_prompt)

### Ejemplo 2: Análisis de noticias de la industria

In [ ]:
user_prompt = "Busca las últimas noticias sobre Azure AI y Microsoft en el sector de inteligencia artificial"

send_prompt_to_agent(user_prompt)

### Ejemplo 3: Investigación de productos o tecnologías

In [ ]:
user_prompt = "Investiga qué es Azure AI Agents SDK y cuáles son sus principales características y casos de uso"

send_prompt_to_agent(user_prompt)

### Ejemplo 4: Análisis competitivo

In [ ]:
user_prompt = "Compara las plataformas de agentes de IA de los principales proveedores cloud (Azure, AWS, Google Cloud)"

send_prompt_to_agent(user_prompt)

### Ejemplo 5: Investigación personalizada

Puedes modificar este prompt con tu propia consulta de investigación:

In [ ]:
# Personaliza esta consulta con tu propia investigación
user_prompt = "Busca información sobre [TU TEMA AQUÍ]"

# Descomenta la siguiente línea para ejecutar tu consulta personalizada
# send_prompt_to_agent(user_prompt)

### Obtener el historial de la conversación

Una vez que hayas terminado de interactuar con el agente, puedes ver el registro completo de la conversación.

In [ ]:
print("\n" + "="*80)
print("HISTORIAL COMPLETO DE LA CONVERSACIÓN")
print("="*80 + "\n")

messages = agent_client.messages.list(thread_id=thread.id, order=ListSortOrder.ASCENDING)
for i, message in enumerate(messages, 1):
    if message.text_messages:
        last_msg = message.text_messages[-1]
        print(f"[{i}] {message.role.upper()}:")
        print("-" * 80)
        print(f"{last_msg.text.value}")
        print("\n")

### Limpieza

Finalmente, elimina el agente para liberar recursos.

In [ ]:
print(f"Eliminando agente (id: {agent.id})...")
agent_client.delete_agent(agent.id)
print("Agente eliminado exitosamente")

## Conclusión

En este ejercicio has aprendido a:

1. ✅ Configurar y usar **BingGroundingTool** con Azure AI Agents SDK
2. ✅ Crear un agente de IA que puede buscar información en Internet en tiempo real
3. ✅ Implementar casos de uso de investigación de mercado:
   - Investigación de tendencias tecnológicas
   - Análisis de noticias de la industria
   - Investigación de productos y tecnologías
   - Análisis competitivo
4. ✅ Configurar parámetros de búsqueda (mercado, idioma, frescura de resultados)
5. ✅ Gestionar conversaciones multi-turno con el agente

### Próximos Pasos

Puedes extender este agente para:
- Combinar Bing Search con otras herramientas (Code Interpreter, Functions)
- Crear agentes multi-agente donde uno se especialice en búsqueda de información
- Implementar filtros de búsqueda más específicos por dominio o categoría
- Guardar y procesar los resultados de búsqueda para análisis posteriores
- Integrar con workflows de Agent Framework para orquestación avanzada